# EV bidding on real auction logs (iPinYou)

The engine's demo A/B runs on simulated traffic. This notebook replays **real**
RTB logs instead: the iPinYou 2013 competition data, season 2 (5 advertisers,
6–12 June 2013). Each impression record carries the real market price (the
highest competing bid), the slot's floor price and whether it was clicked.

**Protocol** (as in Zhang et al., *Real-Time Bidding Benchmarking with iPinYou
Dataset*, 2014): replay the requests in time order; a bid wins when it beats the
logged market price and the floor; a win costs the market price and yields the
logged click; bidding stops when the budget is spent.

**Splits, all by time:** fit the CTR models on 6–9 June; choose the model,
calibrate it, set the guardrail and tune each strategy's bid level on 10 June;
report on 11–12 June, which nothing above ever sees.

**v2** adds user tags as features, a model comparison (logistic regression vs
gradient boosting, with and without tags), isotonic calibration, and a
guardrail that falls back to flat bidding for an advertiser on whose tune-day
traffic the model ranks no better than AUC 0.55. The v1 bidder is kept in the
replay as `ev_v1` and must reproduce the v1 results exactly.

**Caveat:** logs exist only for auctions iPinYou won. A bid above iPinYou's
original one could have won auctions that were never logged. Every published
offline RTB evaluation on this data shares this limitation.

Add the dataset `lastsummer/ipinyou` as an input before running.

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from pandas.api.types import union_categoricals
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# Find the dataset wherever Kaggle mounted it.
ROOT = next(r for r, dirs, _ in os.walk("/kaggle/input") if "training2nd" in dirs)
SEASON = os.path.join(ROOT, "training2nd")
OUT = "/kaggle/working"
print("dataset root:", ROOT)
print(open(os.path.join(ROOT, "known.data.bugs.txt"), errors="replace").read()[:3000])

# Column layout of the season 2/3 logs (Table 1 of the benchmark paper).
COLS = ["bidid", "timestamp", "logtype", "ipinyouid", "useragent", "IP", "region",
        "city", "adexchange", "domain", "url", "urlid", "slotid", "slotwidth",
        "slotheight", "slotvisibility", "slotformat", "slotprice", "creative",
        "bidprice", "payprice", "keypage", "advertiser", "usertag"]
FIT_DAYS, TUNE_DAYS, TEST_DAYS = ["06", "07", "08", "09"], ["10"], ["11", "12"]

In [ ]:
# Read one file's first rows raw, to check the layout before reading 12M rows.
first = sorted(glob.glob(os.path.join(SEASON, "imp.*.txt.bz2")))[0]
peek = pd.read_csv(first, sep="\t", header=None, nrows=3, quoting=3, dtype=str)
assert peek.shape[1] == len(COLS), f"expected {len(COLS)} columns, got {peek.shape[1]}"
peek.columns = COLS
peek.T

In [ ]:
def device_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    tablet = ua.str.contains("ipad|tablet") | (ua.str.contains("android") & ~ua.str.contains("mobile"))
    mobile = ua.str.contains("iphone|ipod|mobile|windows phone|symbian|blackberry")
    return pd.Series(np.select([tablet, mobile], ["tablet", "mobile"], "desktop"), index=ua.index)

def os_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("windows"), ua.str.contains("android"),
         ua.str.contains("iphone|ipad|ipod"), ua.str.contains("mac os"),
         ua.str.contains("linux")],
        ["windows", "android", "ios", "mac", "linux"], "other"), index=ua.index)

def browser_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("msie|trident"), ua.str.contains("chrome"),
         ua.str.contains("firefox"), ua.str.contains("safari"), ua.str.contains("opera")],
        ["ie", "chrome", "firefox", "safari", "opera"], "other"), index=ua.index)

clicked = set()
for f in sorted(glob.glob(os.path.join(SEASON, "clk.*.txt.bz2"))):
    clicked.update(pd.read_csv(f, sep="\t", header=None, usecols=[0], quoting=3,
                               dtype=str)[0])
print(f"{len(clicked):,} clicked bid ids")

CATS = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
        "slotvisibility", "slotformat", "device", "os", "browser", "hour", "day"]
# User tags: a comma-separated list of interest segments per request, known at
# bid time. Kept as a sparse multi-hot matrix beside the frame, row-aligned.
TAG_INDEX, tag_rows, tag_cols, offset = {}, [], [], 0
frames = []
for f in sorted(glob.glob(os.path.join(SEASON, "imp.*.txt.bz2"))):
    t0 = time.time()
    d = pd.read_csv(f, sep="\t", header=None, names=COLS, quoting=3, dtype=str,
                    usecols=["bidid", "timestamp", "useragent", "region", "city",
                             "adexchange", "domain", "slotwidth", "slotheight",
                             "slotvisibility", "slotformat", "slotprice",
                             "payprice", "advertiser", "usertag"])
    out = pd.DataFrame({
        "ts": d["timestamp"].str[:14],                       # yyyymmddHHMMSS
        "click": d["bidid"].isin(clicked).astype("int8"),
        "floor": pd.to_numeric(d["slotprice"], errors="coerce").fillna(0).astype("int32"),
        "pay": pd.to_numeric(d["payprice"], errors="coerce").fillna(0).astype("int32"),
    })
    out["hour"] = d["timestamp"].str[8:10]
    out["day"] = d["timestamp"].str[6:8]
    out["slotsize"] = d["slotwidth"] + "x" + d["slotheight"]
    out["device"], out["os"], out["browser"] = (
        device_of(d["useragent"]), os_of(d["useragent"]), browser_of(d["useragent"]))
    for c in ["advertiser", "adexchange", "region", "city", "domain",
              "slotvisibility", "slotformat"]:
        out[c] = d[c].fillna("na")
    for c in CATS:
        out[c] = out[c].astype("category")
    r, k = [], []
    for i, t in enumerate(d["usertag"].fillna("").to_numpy()):
        if t and t != "null":
            for tag in t.split(","):
                tag = tag.strip()
                if tag:
                    r.append(i)
                    k.append(TAG_INDEX.setdefault(tag, len(TAG_INDEX)))
    tag_rows.append(np.asarray(r, dtype=np.int64) + offset)
    tag_cols.append(np.asarray(k, dtype=np.int64))
    offset += len(d)
    frames.append(out)
    print(f"{os.path.basename(f)}: {len(out):,} rows, {out.click.sum()} clicks, {time.time() - t0:.0f}s")

df = pd.DataFrame({
    c: (union_categoricals([f[c] for f in frames]) if c in CATS
        else np.concatenate([f[c].to_numpy() for f in frames]))
    for c in frames[0].columns})
del frames
rows_, cols_ = np.concatenate(tag_rows), np.concatenate(tag_cols)
TAGS = sparse.csr_matrix((np.ones(len(rows_), dtype=np.float32), (rows_, cols_)),
                         shape=(offset, len(TAG_INDEX)))
TAGS.data[:] = 1.0          # a tag listed twice is still one tag
del rows_, cols_, tag_rows, tag_cols
df["row"] = np.arange(len(df))
df = df.sort_values("ts", kind="stable").reset_index(drop=True)
TAGS = TAGS[df["row"].to_numpy()]
df = df.drop(columns="row")
print(f"{len(TAG_INDEX)} user tags, present on {np.mean(np.diff(TAGS.indptr) > 0):.0%} of impressions")

# Sanity check against Table 3 of the paper (e.g. 1458: 3,083,056 imps, 2,454 clicks).
summary = df.groupby("advertiser", observed=True).agg(
    imps=("click", "size"), clicks=("click", "sum"), cost=("pay", "sum"))
summary["ctr"] = summary.clicks / summary.imps
summary["cpm"] = summary.cost / summary.imps
print(summary.to_string())
print(df.groupby("day", observed=True).size().to_string())

In [ ]:
# ── CTR models: four candidates, one chosen on the tune day ──────────────────
# Bid-time features only. Excluded on purpose: the market price (unknown until
# after the auction), the user and IP ids (memorisation, not generalisation)
# and the creative.
FEATURES = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
            "slotvisibility", "slotformat", "device", "os", "browser", "hour"]
fit_m = df.day.isin(FIT_DAYS).to_numpy()
tune_m = df.day.isin(TUNE_DAYS).to_numpy()
test_m = df.day.isin(TEST_DAYS).to_numpy()
fit, tune, test = (df[m].reset_index(drop=True) for m in (fit_m, tune_m, test_m))
TAGS_FIT, TAGS_TUNE, TAGS_TEST = TAGS[fit_m], TAGS[tune_m], TAGS[test_m]
print(f"fit {len(fit):,} ({fit.click.sum()} clicks)  tune {len(tune):,} ({tune.click.sum()})  "
      f"test {len(test):,} ({test.click.sum()})")

ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, dtype=np.float32)
ohe.fit(fit[FEATURES].astype(str))

def lr_design(frame, tags, with_tags):
    X = ohe.transform(frame[FEATURES].astype(str))
    return sparse.hstack([X, tags], format="csr") if with_tags else X

# Gradient boosting takes integer codes, ranked by frequency on the fit days. A
# feature with few levels is declared categorical; one with thousands (domain,
# city) keeps its 1,000 most frequent values as ranked codes. Anything unseen
# or rarer shares one code.
GBT_LEVELS = {c: fit[c].astype(str).value_counts().index[:1000] for c in FEATURES}
CATEGORICAL = [len(GBT_LEVELS[c]) < 250 for c in FEATURES]

def gbt_design(frame, tags, with_tags):
    cols = []
    for c in FEATURES:
        codes = pd.Categorical(frame[c].astype(str),
                               categories=GBT_LEVELS[c]).codes.astype(np.float32)
        codes[codes < 0] = len(GBT_LEVELS[c])
        cols.append(codes)
    X = np.column_stack(cols)
    return np.hstack([X, tags.toarray()]) if with_tags else X

CANDIDATES = {"lr_base": ("lr", False), "lr_tags": ("lr", True),
              "gbt_base": ("gbt", False), "gbt_tags": ("gbt", True)}
preds, model_rows = {}, []
for name, (kind, with_tags) in CANDIDATES.items():
    t0 = time.time()
    if kind == "lr":
        design, model = lr_design, LogisticRegression(C=0.1, max_iter=500)
    else:
        design = gbt_design
        model = HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.1, l2_regularization=1.0, random_state=0,
            categorical_features=np.array(
                CATEGORICAL + [False] * (len(TAG_INDEX) if with_tags else 0)))
    model.fit(design(fit, TAGS_FIT, with_tags), fit.click)
    p_tune = model.predict_proba(design(tune, TAGS_TUNE, with_tags))[:, 1]
    p_test = model.predict_proba(design(test, TAGS_TEST, with_tags))[:, 1]
    preds[name] = (p_tune, p_test)
    model_rows.append(dict(
        model=name, seconds=round(time.time() - t0),
        tune_auc=roc_auc_score(tune.click, p_tune), tune_logloss=log_loss(tune.click, p_tune),
        test_auc=roc_auc_score(test.click, p_test), test_logloss=log_loss(test.click, p_test)))
    print(model_rows[-1])
model_table = pd.DataFrame(model_rows)
pd.set_option("display.float_format", "{:.5f}".format)
print(model_table.to_string(index=False))
# Chosen by tune-day log loss: it rewards ranking and calibration together, and
# the test days play no part. Their columns above are reported, not used.
BEST = model_table.loc[model_table.tune_logloss.idxmin(), "model"]
print("chosen on the tune day:", BEST)

In [ ]:
# ── Calibration: isotonic regression fitted on the tune day ──────────────────
# The engine bids pCTR × value per click, so a pCTR that is 1.6× too high in
# the top decile is a bid 1.6× too high. Isotonic regression is monotone, so it
# keeps the ranking while correcting the probabilities. Platt scaling is shown
# beside it for comparison; isotonic was chosen in advance.
raw_tune, raw_test = preds[BEST]
iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1.0).fit(raw_tune, tune.click)

def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))

platt = LogisticRegression(C=1e6).fit(logit(raw_tune).reshape(-1, 1), tune.click)
y_test = test.click.to_numpy()
versions = {"raw": raw_test,
            "platt": platt.predict_proba(logit(raw_test).reshape(-1, 1))[:, 1],
            "isotonic": iso.predict(raw_test)}

def deciles(p):
    return np.array_split(np.argsort(p, kind="stable"), 10)

def ece(y, p):
    """Expected calibration error over pCTR deciles."""
    return sum(len(c) / len(p) * abs(p[c].mean() - y[c].mean()) for c in deciles(p))

calibration = pd.DataFrame([dict(
    version=k, auc=roc_auc_score(y_test, p), logloss=log_loss(y_test, p),
    brier=brier_score_loss(y_test, p), ece=ece(y_test, p),
    ece_relative=ece(y_test, p) / y_test.mean(), mean_pctr_over_ctr=p.mean() / y_test.mean())
    for k, p in versions.items()])
print(calibration.to_string(index=False))
for k in ("raw", "isotonic"):
    p = versions[k]
    print(f"\n{k}: mean pCTR vs CTR by decile")
    print(pd.DataFrame([dict(decile=i, mean_pctr=p[c].mean(), ctr=y_test[c].mean())
                        for i, c in enumerate(deciles(p))]).to_string(index=False))
P_TUNE_CAL, P_TEST_CAL = iso.predict(raw_tune), versions["isotonic"]

In [ ]:
# ── Guardrail: no bids from a model that cannot rank this advertiser ─────────
# Fixed in advance, before any result: an advertiser on whose tune-day traffic
# the chosen model ranks below AUC 0.55 bids flat instead. A model that cannot
# tell good impressions from bad only adds noise to the bid.
GUARD_AUC = 0.55
guard_rows = []
for adv, g in tune.groupby("advertiser", observed=True):
    t = test[test.advertiser == adv]
    guard_rows.append(dict(advertiser=adv,
                           tune_auc=roc_auc_score(g.click, raw_tune[g.index]),
                           test_auc=roc_auc_score(t.click, raw_test[t.index])))
guard = pd.DataFrame(guard_rows)
guard["guarded"] = guard.tune_auc < GUARD_AUC
GUARDED = set(guard.loc[guard.guarded, "advertiser"])
print(guard.to_string(index=False))
print("bids flat:", sorted(GUARDED) or "none")

In [ ]:
# ── Replay ───────────────────────────────────────────────────────────────────
# The win rule has no separate floor check. The market price already includes
# the reserve (a winner never pays below the floor), and the paper notes the
# floor column was rescaled, so comparing a bid against it would mix units.
print("rows with market price below the logged floor:",
      f"{(test.pay < test.floor).mean():.2%}")

def replay(bids, pay, clicks, budget):
    """Requests in time order: win when the bid beats the market price, pay the
    market price, stop once the budget is spent."""
    win = bids > pay
    cost = np.where(win, pay, 0)
    won = win & (np.cumsum(cost) <= budget)
    return int(won.sum()), int(clicks[won].sum()), int(cost[won].sum())

tune["p_v1"], test["p_v1"] = preds["lr_base"]
tune["p_cal"], test["p_cal"] = P_TUNE_CAL, P_TEST_CAL

def by_advertiser(frame):
    return {adv: {"pay": g["pay"].to_numpy(), "click": g["click"].to_numpy(),
                  "p_v1": g["p_v1"].to_numpy(), "p_cal": g["p_cal"].to_numpy(),
                  "block": (g["day"].astype(str) + g["hour"].astype(str)).to_numpy()}
            for adv, g in frame.groupby("advertiser", observed=True)}

TUNE, TEST = by_advertiser(tune), by_advertiser(test)

# flat:     one bid for every request (the engine's flat CPM)
# ev_v1:    v1's bidder, base-feature logistic regression, uncalibrated
# ev:       the chosen model, calibrated
# ev_guard: ev, except advertisers the guardrail caught bid flat
# An EV bid is level * pCTR / mean pCTR, i.e. pCTR × a value per click, so the
# level is the bid on an average request (the paper's linear bidder). The mean
# is the tune day's, fixed before the test period.
STRATEGIES = ["flat", "ev_v1", "ev", "ev_guard"]
SIGNAL = {"ev_v1": "p_v1", "ev": "p_cal", "ev_guard": "p_cal"}
NORM = {(s, adv): a[SIGNAL[s]].mean() for s in SIGNAL for adv, a in TUNE.items()}

def bids_for(strategy, a, adv, level):
    if strategy == "flat" or (strategy == "ev_guard" and adv in GUARDED):
        return np.full(len(a["pay"]), float(level))
    return level * a[SIGNAL[strategy]] / NORM[strategy, adv]

GRID = np.arange(2, 402, 2)
BUDGET_FRACTIONS = {"1/32": 1 / 32, "1/8": 1 / 8, "1/2": 1 / 2}

def run(a, adv, strategy, level, fraction):
    return replay(bids_for(strategy, a, adv, level), a["pay"], a["click"],
                  fraction * a["pay"].sum())

# Each strategy gets its own best bid level, chosen on the tune day: the most
# clicks for the budget, ties to the cheaper. Same grid, same rule, every arm.
levels = {}
for adv, a in TUNE.items():
    for s in STRATEGIES:
        for name, frac in BUDGET_FRACTIONS.items():
            scores = [(run(a, adv, s, b, frac), b) for b in GRID]
            levels[adv, s, name] = max(scores, key=lambda x: (x[0][1], -x[0][2]))[1]

rows = []
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            imps, clk, cost = run(a, adv, s, levels[adv, s, name], frac)
            rows.append(dict(advertiser=adv, budget=name, strategy=s,
                             level=int(levels[adv, s, name]), imps=imps, clicks=clk, cost=cost))
results = pd.DataFrame(rows)
results["ecpc"] = results.cost / results.clicks.replace(0, np.nan)
print(results.pivot_table(index=["advertiser", "budget"], columns="strategy",
                          values="clicks").to_string())

In [ ]:
# ── Uncertainty: block bootstrap over test hours ─────────────────────────────
# Requests within an hour are correlated (same campaigns, same traffic), so
# whole hours are resampled, kept in time order, with the budget rescaled to the
# resample's own cost. Every strategy replays the SAME resample: the comparison
# is paired, and the tuned bid levels stay fixed.
rng = np.random.default_rng(0)
B = 300
BLOCKS = {adv: [np.flatnonzero(a["block"] == k) for k in np.unique(a["block"])]
          for adv, a in TEST.items()}   # np.unique sorts, so blocks are in time order

def pooled(sample):
    """Clicks and cost per (budget, strategy), summed over advertisers."""
    out = {}
    for adv, idx in sample.items():
        a = {k: TEST[adv][k][idx] for k in ("pay", "click", "p_v1", "p_cal")}
        for name, frac in BUDGET_FRACTIONS.items():
            for s in STRATEGIES:
                _, clk, cost = run(a, adv, s, levels[adv, s, name], frac)
                out[name, s, "clicks"] = out.get((name, s, "clicks"), 0) + clk
                out[name, s, "cost"] = out.get((name, s, "cost"), 0) + cost
    return out

observed = pooled({adv: np.concatenate(b) for adv, b in BLOCKS.items()})
boot = []
for _ in range(B):
    boot.append(pooled({adv: np.concatenate([b[i] for i in np.sort(
        rng.integers(0, len(b), len(b)))]) for adv, b in BLOCKS.items()}))

def lift(o, name, s):         # extra clicks over flat bidding, same budget
    return o[name, s, "clicks"] / o[name, "flat", "clicks"] - 1

def ecpc_change(o, name, s):  # change in cost per click against flat bidding
    return ((o[name, s, "cost"] / o[name, s, "clicks"])
            / (o[name, "flat", "cost"] / o[name, "flat", "clicks"]) - 1)

summary_rows = []
for name in BUDGET_FRACTIONS:
    for s in STRATEGIES[1:]:
        lifts = np.array([lift(o, name, s) for o in boot])
        ecpcs = np.array([ecpc_change(o, name, s) for o in boot])
        summary_rows.append(dict(
            budget=name, strategy=s,
            flat_clicks=observed[name, "flat", "clicks"], clicks=observed[name, s, "clicks"],
            click_lift=lift(observed, name, s),
            lift_ci=[float(np.quantile(lifts, 0.025)), float(np.quantile(lifts, 0.975))],
            ecpc_change=ecpc_change(observed, name, s),
            ecpc_ci=[float(np.quantile(ecpcs, 0.025)), float(np.quantile(ecpcs, 0.975))],
        ))
summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.float_format", "{:.3f}".format)
print(summary_df.to_string(index=False))

# v1 published these: ev_v1 must match them exactly, or the pipeline changed.
V1 = {"1/32": (127, 423), "1/8": (406, 694), "1/2": (1562, 1726)}
for name, (flat, ev) in V1.items():
    got = (observed[name, "flat", "clicks"], observed[name, "ev_v1", "clicks"])
    print(f"v1 check {name}: expected {(flat, ev)}, got {got}", "OK" if got == (flat, ev) else "MISMATCH")

In [ ]:
# ── Robustness: is any gap an artefact of tuning on one noisy day? ───────────
# Every strategy gets its best possible level in hindsight, picked on the test
# days themselves. That flatters all of them equally: a gap that survives comes
# from the bidding rule, not from a level being badly chosen.
hindsight = {}
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            best = max((run(a, adv, s, b, frac) for b in GRID), key=lambda r: (r[1], -r[2]))
            hindsight[name, s] = hindsight.get((name, s), 0) + best[1]
for name in BUDGET_FRACTIONS:
    print(f"budget {name}: " + ", ".join(
        f"{s} {hindsight[name, s]}" + ("" if s == "flat" else
                                        f" ({hindsight[name, s] / hindsight[name, 'flat'] - 1:+.1%})")
        for s in STRATEGIES))

In [ ]:
with open(f"{OUT}/ipinyou_replay_results.json", "w") as f:
    json.dump({
        "version": 2,
        "data": "iPinYou season 2 (training2nd), 5 advertisers, 6-12 June 2013",
        "splits": {"fit": FIT_DAYS, "tune": TUNE_DAYS, "test": TEST_DAYS},
        "rows": {"fit": int(len(fit)), "tune": int(len(tune)), "test": int(len(test))},
        "clicks": {"fit": int(fit.click.sum()), "tune": int(tune.click.sum()),
                   "test": int(test.click.sum())},
        "user_tags": len(TAG_INDEX),
        "models": model_table.to_dict("records"),
        "chosen_model": BEST,
        "calibration": {"method": "isotonic, fitted on the tune day",
                        "test": calibration.to_dict("records")},
        "guardrail": {"min_tune_auc": GUARD_AUC, "advertisers": guard.to_dict("records")},
        "bootstrap": {"blocks": "test hours", "replicates": B},
        "summary": summary_df.to_dict("records"),
        "hindsight_clicks": [{"budget": n, "strategy": s, "clicks": hindsight[n, s]}
                             for n in BUDGET_FRACTIONS for s in STRATEGIES],
        "per_advertiser": results.to_dict("records"),
    }, f, indent=2, default=float)
print("wrote ipinyou_replay_results.json")

In [ ]:
# ── Export: the model the engine serves, and real requests to replay ─────────
# 1. ipinyou_ctr_model.json: the chosen model as plain JSON. Per-feature
#    weights keyed by the exact strings used above, the weight of each
#    feature's "infrequent" bucket (where unseen values land too, as in
#    OneHotEncoder's infrequent_if_exist), the user-tag weights, the isotonic
#    calibration as its breakpoints, and the bid levels tuned for the 1/2
#    budget. The engine scores it in pure Python: no pickle, no scikit-learn.
# 2. ipinyou_replay_sample.csv.gz: real test-day requests with the pCTR this
#    notebook gives each one. The engine's tests rescore every row and must
#    agree: a train/serve parity check. Bid ids, user ids, IPs and URLs are
#    left out.
import hashlib

kind, with_tags = CANDIDATES[BEST]
assert kind == "lr" and with_tags, "the engine serves logistic regression with user tags"
# The loop above kept predictions, not models, so refit the chosen one. Same
# data, same solver: it must reproduce the predictions that were evaluated.
serve = LogisticRegression(C=0.1, max_iter=500).fit(lr_design(fit, TAGS_FIT, True), fit.click)
assert np.allclose(serve.predict_proba(lr_design(test, TAGS_TEST, True))[:, 1],
                   preds[BEST][1], rtol=1e-6), "the refit does not reproduce the evaluated model"

names = ohe.get_feature_names_out()
coef = serve.coef_[0]
weights, infrequent = {f: {} for f in FEATURES}, {}
for name, w in zip(names, coef[:len(names)]):
    feature = max((f for f in FEATURES if name.startswith(f + "_")), key=len)
    value = name[len(feature) + 1:]
    if value == "infrequent_sklearn":
        infrequent[feature] = float(w)
    else:
        weights[feature][value] = float(w)
tag_names = sorted(TAG_INDEX, key=TAG_INDEX.get)
tag_weights = {t: float(w) for t, w in zip(tag_names, coef[len(names):])}
assert len(tag_weights) == len(coef) - len(names)

scoring = {"intercept": float(serve.intercept_[0]), "weights": weights,
           "infrequent": infrequent, "tag_weights": tag_weights,
           "calibration": {"method": "isotonic", "x": iso.X_thresholds_.tolist(),
                           "y": iso.y_thresholds_.tolist()}}
version = hashlib.sha256(json.dumps(scoring, sort_keys=True).encode()).hexdigest()[:12]
model_json = {
    "name": "ipinyou-lr-usertags",
    "version": version,
    "trained_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": "iPinYou RTB dataset, season 2 (Zhang et al., Real-Time Bidding "
              "Benchmarking with iPinYou Dataset, 2014)",
    "splits": {"fit": FIT_DAYS, "tune": TUNE_DAYS, "test": TEST_DAYS},
    "features": FEATURES,
    **scoring,
    "bidding": {
        "budget": "1/2",
        "rule": "bid = level * pCTR / mean_pctr; a bid wins when it beats the market price",
        "advertisers": {adv: {"ev_level": int(levels[adv, "ev", "1/2"]),
                              "flat_level": int(levels[adv, "flat", "1/2"]),
                              "mean_pctr": float(NORM["ev", adv])} for adv in TEST},
    },
    "price_unit": "CNY fen per thousand impressions (CPM)",
    "metrics": {"models": model_table.to_dict("records"),
                "calibration": calibration.to_dict("records")},
    "offline_evaluation": {
        "summary": summary_df.to_dict("records"),
        "hindsight_clicks": [{"budget": n, "strategy": s, "clicks": hindsight[n, s]}
                             for n in BUDGET_FRACTIONS for s in STRATEGIES],
        "test_auctions": int(len(test)), "test_clicks": int(test.click.sum()),
    },
}
with open(f"{OUT}/ipinyou_ctr_model.json", "w") as f:
    json.dump(model_json, f, default=float)

rng = np.random.default_rng(1)
uniform = rng.choice(len(test), 20_000, replace=False)
clicked = np.flatnonzero(test.click.to_numpy() == 1)
idx = np.union1d(uniform, clicked)
indptr, indices = TAGS_TEST.indptr, TAGS_TEST.indices
sample = pd.DataFrame({f: test[f].astype(str).to_numpy()[idx] for f in FEATURES})
sample["usertags"] = [",".join(tag_names[j] for j in indices[indptr[i]:indptr[i + 1]]) for i in idx]
sample["market_price"] = test["pay"].to_numpy()[idx]
sample["floor"] = test["floor"].to_numpy()[idx]
sample["click"] = test["click"].to_numpy()[idx]
sample["day"] = test["day"].astype(str).to_numpy()[idx]
# "clicked" rows were added so a visitor can find some; they are not a random
# draw, and CTR measured on this file is not the real CTR.
sample["sampled_as"] = np.where(np.isin(idx, uniform), "uniform", "clicked")
sample["pctr_raw"] = preds[BEST][1][idx]
sample["pctr_cal"] = P_TEST_CAL[idx]
sample["pctr_percentile"] = test["p_cal"].rank(pct=True).to_numpy()[idx]
sample.to_csv(f"{OUT}/ipinyou_replay_sample.csv.gz", index=False, compression="gzip",
              float_format="%.17g")

print(f"model {model_json['name']} {version}: {sum(map(len, weights.values()))} feature weights, "
      f"{len(infrequent)} infrequent buckets, {len(tag_weights)} tags, "
      f"{len(iso.X_thresholds_)} calibration points")
print(f"sample: {len(sample):,} requests ({(sample.sampled_as == 'uniform').sum():,} uniform, "
      f"{sample.click.sum():,} clicked), "
      f"{os.path.getsize(f'{OUT}/ipinyou_replay_sample.csv.gz') / 2**20:.1f} MB")